# 🚦 Traffic Demand Prediction — FIXED Pipeline v2

## 🔑 Root Cause of Previous Score=0

The original notebook had **3 critical bugs**:

| Bug | What Happened | Fix |
|-----|--------------|-----|
| ❌ Wrong CV strategy | KFold shuffled day48+day49 together, making geo+slot features leak | ✅ Use day48→day49 time-based split |
| ❌ Complex KFold target encoding | Failed silently in Colab, fed garbage features to models | ✅ Direct lookup dict (simple, fast, reliable) |
| ❌ Wrong validation proxy | CV score didn't reflect test distribution | ✅ day48=train, day49=validation, day49 test=predict |

**Key data insight discovered:**  
- Test is 100% `day=49`  
- **Zero test (geohash,slot) combos appear in training day=49**  
- 88.9% of test combos come from training day=48  
- So the real task is: **learn patterns from day48 → predict day49**

**Expected scores:** Baseline lookup=52 → LightGBM=59 → target 65+


## 1. Install & Imports

In [ ]:
!pip install lightgbm xgboost catboost optuna -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings; warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')
print("✅ Imports done")


## 2. Load Data

In [ ]:
train = pd.read_csv('/content/train.csv')
test  = pd.read_csv('/content/test.csv')

def parse_slot(ts):
    h, m = ts.split(':')
    return int(h) * 4 + int(m) // 15   # 96 slots per day

for df in [train, test]:
    df['slot'] = df['timestamp'].apply(parse_slot)
    df['hour'] = df['timestamp'].apply(lambda x: int(x.split(':')[0]))

print(f"Train: {train.shape}  Test: {test.shape}")
print(f"Train days: {train['day'].value_counts().to_dict()}")
print(f"Test  days: {test['day'].value_counts().to_dict()}")
train.head(3)


## 3. KEY DATA INSIGHT — Why Test is Structurally Different

In [ ]:
train48 = train[train['day']==48]
train49 = train[train['day']==49]
test_combos    = set(zip(test['geohash'],    test['slot']))
train49_combos = set(zip(train49['geohash'], train49['slot']))
train48_combos = set(zip(train48['geohash'], train48['slot']))
all_combos     = set(zip(train['geohash'],   train['slot']))

print("=== GEO+SLOT COMBO COVERAGE ===")
print(f"Train48:  {len(train48):6d} rows  {len(train48_combos):5d} unique (geo,slot) combos")
print(f"Train49:  {len(train49):6d} rows  {len(train49_combos):5d} unique (geo,slot) combos")
print(f"Test:     {len(test):6d} rows  {len(test_combos):5d} unique (geo,slot) combos")
print()
print(f"Test combos in Train49: {len(test_combos & train49_combos)} ({100*len(test_combos & train49_combos)/len(test_combos):.1f}%)")
print(f"Test combos in Train48: {len(test_combos & train48_combos)} ({100*len(test_combos & train48_combos)/len(test_combos):.1f}%)")
print()
print("CONCLUSION: Test is day=49, train day=49 has ZERO overlap with test combos.")
print("The real task: learn from day48 patterns → predict day49 demand.")


## 4. EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Demand distribution
axes[0,0].hist(train['demand'], bins=60, color='steelblue', edgecolor='white')
axes[0,0].set_title('Demand Distribution')

axes[0,1].hist(np.log1p(train['demand']), bins=60, color='salmon', edgecolor='white')
axes[0,1].set_title('log1p(Demand)')

# Hourly pattern
hourly = train.groupby('hour')['demand'].mean()
axes[0,2].plot(hourly.index, hourly.values, marker='o', color='royalblue')
axes[0,2].fill_between(hourly.index, hourly.values, alpha=0.2)
axes[0,2].set_title('Mean Demand by Hour')
axes[0,2].set_xlabel('Hour')

# Day comparison
axes[1,0].hist(train48['demand'], bins=40, alpha=0.6, label='Day 48', color='green')
axes[1,0].hist(train49['demand'], bins=40, alpha=0.6, label='Day 49', color='red')
axes[1,0].legend(); axes[1,0].set_title('Day 48 vs Day 49 Demand')

# Road type vs demand
road_demand = train.groupby('RoadType')['demand'].mean().sort_values()
axes[1,1].barh(road_demand.index, road_demand.values, color='teal')
axes[1,1].set_title('Mean Demand by RoadType')

# Weather vs demand
wx_demand = train.groupby('Weather')['demand'].mean().sort_values()
axes[1,2].barh(wx_demand.index, wx_demand.values, color='coral')
axes[1,2].set_title('Mean Demand by Weather')

plt.tight_layout(); plt.show()
print(f"Demand skew: {train['demand'].skew():.3f}")
print(f"Day 48 mean: {train48['demand'].mean():.4f}  Day 49 mean: {train49['demand'].mean():.4f}")


## 5. Feature Engineering

In [ ]:
def build_features(target_df, source_df):
    """
    Build features for target_df using statistics computed from source_df.
    CRITICAL: source_df must NOT include target_df rows to avoid leakage.
    - For validation: source=train48, target=train49
    - For test:       source=full train, target=test
    """
    gm  = source_df['demand'].mean()
    
    # Geo+slot target encodings (the most powerful features)
    gsl  = source_df.groupby(['geohash','slot'])['demand'].mean().to_dict()
    gl   = source_df.groupby('geohash')['demand'].mean().to_dict()
    sl   = source_df.groupby('slot')['demand'].mean().to_dict()
    hl   = source_df.groupby('hour')['demand'].mean().to_dict()
    
    s2 = source_df.copy()
    s2['geo4'] = s2['geohash'].str[:4]
    s2['geo5'] = s2['geohash'].str[:5]
    g4l  = s2.groupby('geo4')['demand'].mean().to_dict()
    g5l  = s2.groupby('geo5')['demand'].mean().to_dict()
    g4sl = s2.groupby(['geo4','slot'])['demand'].mean().to_dict()
    
    d = target_df.copy()
    d['geo4'] = d['geohash'].str[:4]
    d['geo5'] = d['geohash'].str[:5]
    
    # Primary feature: exact geohash×slot mean (with hierarchical fallback)
    d['geo_slot_te']  = [gsl.get((g,s), gl.get(g, sl.get(s, gm)))
                         for g,s in zip(d['geohash'], d['slot'])]
    d['geo_mean']     = d['geohash'].map(gl).fillna(gm)
    d['slot_mean']    = d['slot'].map(sl).fillna(gm)
    d['hour_mean']    = d['hour'].map(hl).fillna(gm)
    d['geo4_mean']    = d['geo4'].map(g4l).fillna(gm)
    d['geo5_mean']    = d['geo5'].map(g5l).fillna(gm)
    d['geo4_slot_te'] = [g4sl.get((g,s), g4l.get(g, sl.get(s, gm)))
                         for g,s in zip(d['geo4'], d['slot'])]
    
    # Derived ratio features
    d['geo_slot_resid'] = d['geo_slot_te'] - d['geo_mean']
    d['slot_geo_ratio'] = d['slot_mean'] / (d['geo_mean'] + 1e-6)
    d['geo4_te_diff']   = d['geo_slot_te'] - d['geo4_slot_te']
    
    # Categorical encoding
    binary  = {'Yes':1,'No':0,'Allowed':1,'Not Allowed':0}
    road    = {'Highway':3,'Street':2,'Residential':1}
    weather = {'Sunny':0,'Foggy':1,'Rainy':2,'Snowy':3}
    d['lv_enc']   = d['LargeVehicles'].map(binary).fillna(0)
    d['lm_enc']   = d['Landmarks'].map(binary).fillna(0)
    d['road_enc'] = d['RoadType'].map(road).fillna(0)
    d['wx_enc']   = d['Weather'].map(weather).fillna(-1)
    
    tm = source_df['Temperature'].median()
    d['temp']      = d['Temperature'].fillna(tm)
    d['temp_miss'] = d['Temperature'].isna().astype(int)
    
    # Cyclical time features
    d['hour_sin'] = np.sin(2*np.pi*d['hour']/24)
    d['hour_cos'] = np.cos(2*np.pi*d['hour']/24)
    d['slot_sin'] = np.sin(2*np.pi*d['slot']/96)
    d['slot_cos'] = np.cos(2*np.pi*d['slot']/96)
    
    # Interaction features
    d['lane_road'] = d['NumberofLanes'] * d['road_enc']
    d['lane_lv']   = d['NumberofLanes'] * d['lv_enc']
    
    return d

FEATS = [
    'geo_slot_te', 'geo_mean', 'slot_mean', 'hour_mean',
    'geo4_mean', 'geo5_mean', 'geo4_slot_te',
    'geo_slot_resid', 'slot_geo_ratio', 'geo4_te_diff',
    'hour', 'slot', 'NumberofLanes',
    'lv_enc', 'lm_enc', 'road_enc', 'wx_enc',
    'temp', 'temp_miss', 'lane_road', 'lane_lv',
    'hour_sin', 'hour_cos', 'slot_sin', 'slot_cos',
]

train48 = train[train['day']==48].copy()
train49 = train[train['day']==49].copy()

# Validation set (leak-free): train48→train49
val_f = build_features(train49, train48)
tr48_f = build_features(train48, train48)

print(f"Features: {len(FEATS)}")
print(f"Train48 features shape: {tr48_f[FEATS].shape}")
print(f"Val  (day49) shape:     {val_f[FEATS].shape}")
print("✅ Feature engineering done")


## 6. Baseline — Pure Geo+Slot Lookup

In [ ]:
# Sanity check: how well does simple lookup do?
baseline_preds = val_f['geo_slot_te'].values
baseline_score = max(0, 100 * r2_score(train49['demand'], baseline_preds))
print(f"Pure geo+slot lookup score (day48→day49): {baseline_score:.2f}")
print("This is our floor — every model must beat this.")


## 7. LightGBM — with Proper Validation

In [ ]:
lgb_m = lgb.LGBMRegressor(
    n_estimators=3000, learning_rate=0.02,
    num_leaves=127, min_child_samples=10,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.05, reg_lambda=0.5,
    random_state=SEED, verbose=-1
)
lgb_m.fit(
    tr48_f[FEATS], train48['demand'].values,
    eval_set=[(val_f[FEATS], train49['demand'].values)],
    callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(100)]
)
lgb_val = np.clip(lgb_m.predict(val_f[FEATS]), 0, 1)
lgb_score = max(0, 100 * r2_score(train49['demand'], lgb_val))
print(f"\nLightGBM score (day48→day49): {lgb_score:.2f}  (best_iter={lgb_m.best_iteration_})")


In [ ]:
# Feature importance
fi = pd.DataFrame({'feature': FEATS, 'importance': lgb_m.feature_importances_})
fi = fi.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(fi['feature'][::-1], fi['importance'][::-1], color='steelblue')
plt.title('LightGBM Feature Importance')
plt.tight_layout()
plt.show()
print(fi.head(10).to_string(index=False))


## 8. XGBoost

In [ ]:
xgb_m = xgb.XGBRegressor(
    n_estimators=3000, learning_rate=0.02, max_depth=7,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.05, reg_lambda=0.5,
    random_state=SEED, tree_method='hist', verbosity=0,
    early_stopping_rounds=200, eval_metric='rmse'
)
xgb_m.fit(
    tr48_f[FEATS], train48['demand'].values,
    eval_set=[(val_f[FEATS], train49['demand'].values)],
    verbose=False
)
xgb_val = np.clip(xgb_m.predict(val_f[FEATS]), 0, 1)
xgb_score = max(0, 100 * r2_score(train49['demand'], xgb_val))
print(f"XGBoost score (day48→day49): {xgb_score:.2f}  (best_iter={xgb_m.best_iteration})")


## 9. CatBoost

In [ ]:
cat_m = CatBoostRegressor(
    iterations=3000, learning_rate=0.02,
    depth=8, l2_leaf_reg=3,
    random_seed=SEED, verbose=0,
    early_stopping_rounds=200
)
cat_m.fit(
    tr48_f[FEATS], train48['demand'].values,
    eval_set=(val_f[FEATS], train49['demand'].values)
)
cat_val = np.clip(cat_m.predict(val_f[FEATS]), 0, 1)
cat_score = max(0, 100 * r2_score(train49['demand'], cat_val))
print(f"CatBoost score (day48→day49): {cat_score:.2f}  (best_iter={cat_m.best_iteration_})")


## 10. Ensemble — Find Best Weights

In [ ]:
from scipy.optimize import minimize

val_preds_dict = {'LGB': lgb_val, 'XGB': xgb_val, 'CAT': cat_val}
y_val = train49['demand'].values

# Grid search over weights
best_score, best_w = 0, (1, 0, 0)
for w1 in np.arange(0, 1.1, 0.1):
    for w2 in np.arange(0, 1.1-w1, 0.1):
        w3 = round(1 - w1 - w2, 2)
        if w3 < 0: continue
        ens = w1*lgb_val + w2*xgb_val + w3*cat_val
        s   = max(0, 100 * r2_score(y_val, np.clip(ens, 0, 1)))
        if s > best_score:
            best_score, best_w = s, (w1, w2, w3)

print(f"Individual scores: LGB={lgb_score:.2f}  XGB={xgb_score:.2f}  CAT={cat_score:.2f}")
print(f"Best ensemble: w_lgb={best_w[0]:.1f}  w_xgb={best_w[1]:.1f}  w_cat={best_w[2]:.1f}  score={best_score:.2f}")
W_LGB, W_XGB, W_CAT = best_w


## 11. Optuna Tuning (Optional — Run for Higher Score)

In [ ]:
# Uncomment and run for ~10 min to push score higher
# N_TRIALS = 80  # increase for better tuning

# def objective(trial):
#     params = dict(
#         n_estimators      = trial.suggest_int('n_estimators', 200, 2000),
#         learning_rate     = trial.suggest_float('lr', 0.005, 0.1, log=True),
#         num_leaves        = trial.suggest_int('num_leaves', 31, 255),
#         min_child_samples = trial.suggest_int('min_child', 5, 50),
#         subsample         = trial.suggest_float('sub', 0.5, 1.0),
#         colsample_bytree  = trial.suggest_float('col', 0.5, 1.0),
#         reg_alpha         = trial.suggest_float('alpha', 1e-4, 10.0, log=True),
#         reg_lambda        = trial.suggest_float('lambda', 1e-4, 10.0, log=True),
#         random_state=SEED, verbose=-1
#     )
#     m = lgb.LGBMRegressor(**params)
#     m.fit(tr48_f[FEATS], train48['demand'].values,
#           eval_set=[(val_f[FEATS], train49['demand'].values)],
#           callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
#     p = np.clip(m.predict(val_f[FEATS]), 0, 1)
#     return r2_score(train49['demand'], p)

# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
# print(f"Best val score: {study.best_value*100:.2f}")
# print(study.best_params)
print("Skipping Optuna — using tuned params from best_iter above.")
print("Uncomment the code above to run tuning.")


## 12. Train on Full Data → Generate Test Predictions

In [ ]:
# Retrain on ALL training data (day48 + day49) for test predictions
tr_all = build_features(train, train)
test_f = build_features(test,  train)
y_all  = train['demand'].values

print("Training LightGBM on full train...")
lgb_full = lgb.LGBMRegressor(
    n_estimators=lgb_m.best_iteration_,  # use early-stopped iteration count
    learning_rate=0.02, num_leaves=127,
    min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.05, reg_lambda=0.5, random_state=SEED, verbose=-1
)
lgb_full.fit(tr_all[FEATS], y_all)
t_lgb = np.clip(lgb_full.predict(test_f[FEATS]), 0, 1)
print(f"LGB done: mean={t_lgb.mean():.4f}")

print("Training XGBoost on full train...")
xgb_full = xgb.XGBRegressor(
    n_estimators=xgb_m.best_iteration, learning_rate=0.02, max_depth=7,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.05, reg_lambda=0.5,
    random_state=SEED, tree_method='hist', verbosity=0
)
xgb_full.fit(tr_all[FEATS], y_all)
t_xgb = np.clip(xgb_full.predict(test_f[FEATS]), 0, 1)
print(f"XGB done: mean={t_xgb.mean():.4f}")

print("Training CatBoost on full train...")
cat_full = CatBoostRegressor(
    iterations=cat_m.best_iteration_, learning_rate=0.02,
    depth=8, l2_leaf_reg=3, random_seed=SEED, verbose=0
)
cat_full.fit(tr_all[FEATS], y_all)
t_cat = np.clip(cat_full.predict(test_f[FEATS]), 0, 1)
print(f"CAT done: mean={t_cat.mean():.4f}")

print("\n✅ All full-train models trained")


In [ ]:
# Final blended prediction
t_final = np.clip(W_LGB*t_lgb + W_XGB*t_xgb + W_CAT*t_cat, 0, 1)

# Distribution sanity check
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(train['demand'], bins=60, color='steelblue', edgecolor='white', alpha=0.7, label='Train')
axes[0].hist(t_final, bins=60, color='salmon', edgecolor='white', alpha=0.7, label='Test Pred')
axes[0].legend(); axes[0].set_title('Train vs Test Prediction Distribution')

axes[1].scatter(t_lgb[:5000], t_xgb[:5000], alpha=0.1, s=3)
axes[1].set_xlabel('LGB preds'); axes[1].set_ylabel('XGB preds')
axes[1].set_title('Model Agreement (LGB vs XGB)')
plt.tight_layout(); plt.show()

print(f"Test predictions: mean={t_final.mean():.4f}  std={t_final.std():.4f}  min={t_final.min():.4f}  max={t_final.max():.4f}")
print(f"Train demand:     mean={train['demand'].mean():.4f}  std={train['demand'].std():.4f}")


## 13. Save Submission

In [ ]:
submission = pd.DataFrame({
    'Index' : test['Index'].values,
    'demand': t_final
})

assert submission.shape == (41778, 2), f"Wrong shape: {submission.shape}"
assert not submission.isnull().any().any(), "NaN found in submission!"
assert (submission['demand'] >= 0).all() and (submission['demand'] <= 1).all(), "Values out of [0,1]!"

submission.to_csv('/content/submission.csv', index=False)
print("✅ submission.csv saved")
print(f"Shape: {submission.shape}")
print(submission.head(10).to_string(index=False))


## 14. Next Steps to Push Score Higher

### 🎯 Quick wins (likely +3-5 points each)
1. **More Optuna trials** — uncomment Section 11, set `N_TRIALS=150`
2. **Geohash neighbour features** — decode geohash to lat/lon, compute mean demand of 8 spatial neighbours  
3. **Slot smoothing** — instead of raw mean, use Bayesian smoothing `(count*mean + k*global) / (count+k)`

### 🧠 Architecture improvements
4. **Neural network on residuals** — train MLP to predict `demand - geo_slot_te`  
5. **TabNet** — `pip install pytorch-tabnet` — attention-based table learning
6. **Layer-2 stacking** — add val predictions as new features to a meta-learner

### 📊 Feature ideas
7. **Peak hour × geohash interaction** — `geo_mean * is_morning_peak`
8. **Temperature bins** — discretize temp into cold/mild/hot categories
9. **Road quality score** — `road_enc * NumberofLanes * (1 + LargeVehicles)`
